In [1]:
from openvino import Core
core = Core()
print(core.available_devices)
# Should print: ['CPU', 'GPU', 'NPU']

['CPU', 'GPU', 'NPU']


In [1]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="microsoft/Phi-3.5-mini-instruct",
    local_dir="../models/phi-3.5-mini-instruct-src",
    local_dir_use_symlinks=False,
)


c:\Users\matse\GIG\.venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\matse\GIG\.venv2\Lib\site-packages\huggingface_hub\file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 20 files:  50%|█████     | 10/20 [00:01<00:01,  7.32it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip

'C:\\Users\\matse\\GIG\\models\\phi-3.5-mini-instruct-src'

In [8]:
from __future__ import annotations

import subprocess
from pathlib import Path


def run(cmd: list[str]) -> None:
    print(">", " ".join(cmd))
    p = subprocess.Popen(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="")
    code = p.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}")


def export_quantized_openvino(
    src: str | Path,
    out: str | Path,
    bits: int = 8,
    task: str = "text-generation",
) -> Path:
    """Export a local HF model folder to quantized OpenVINO artifacts.

    Args:
        src: Local source model folder (config + tokenizer + safetensors).
        out: Output folder for OpenVINO model files.
        bits: Quantization bit-width, either 8 or 4.
        task: Optimum export task.

    Returns:
        Resolved output path.
    """
    src_path = Path(src).resolve()
    out_path = Path(out).resolve()
    out_path.mkdir(parents=True, exist_ok=True)

    if not src_path.exists():
        raise FileNotFoundError(f"Source model folder not found: {src_path}")
    if bits not in (8, 4):
        raise ValueError("bits must be 8 or 4")

    weight_format = "int8" if bits == 8 else "int4"

    cmd = [
        "optimum-cli",
        "export",
        "openvino",
        "--model",
        str(src),
        "--weight-format",
        weight_format,
        "--task",
        task,
        str(out_path),
    ]
    run(cmd)

    print("\nDone.")
    print(f"Source: {src}")
    print(f"Output: {out_path}")
    print("Expected files include: openvino_model.xml, openvino_model.bin, tokenizer files")
    return out_path


# Example usage from notebook/script:
# export_quantized_openvino(
#     src="../models/phi-4-mini-instruct-src",
#     out="../models/phi4-ov-int8-local",
#     bits=8,
# )

export_quantized_openvino(
    src="../models/phi-3.5-mini-instruct-src",
    out="../models/phi-3.5-mini-instruct-ov-int4-local-2",
    bits=4
    )

> optimum-cli export openvino --model ../models/phi-3.5-mini-instruct-src --weight-format int4 --task text-generation C:\Users\matse\GIG\models\phi-3.5-mini-instruct-ov-int4-local-2
`torch_dtype` is deprecated! Use `dtype` instead!

Loading checkpoint shards: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 2/2 [00:00<00:00, 11.04it/s]
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
C:\Users\matse\GIG\.venv2\Lib\site-packages\transformers\masking_utils.py:207: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
C:\Users\matse\GIG\.venv2\Lib\site-packages\optimum\exporters\openvino\model_patcher.py:207: TracerWarning: torch.tensor results are registered a

WindowsPath('C:/Users/matse/GIG/models/phi-3.5-mini-instruct-ov-int4-local-2')

In [15]:
from optimum.intel import OVModelForCausalLM
from transformers import AutoTokenizer

model_path = "../models/phi-3.5-mini-instruct-ov-int4-local-2"

# Load with NPU as the device
# HINT: "NPU" routes compute; "CPU" is the fallback for unsupported ops
model = OVModelForCausalLM.from_pretrained(
    model_path,
    device="NPU",  
    use_cache = False, 
    dynamic_shapes=False,        # Route to Intel NPU
    ov_config={
        "PERFORMANCE_HINT": "LATENCY",   # Optimize for fast first token
        "NUM_STREAMS": "1",              # Single stream = lower latency
        "CACHE_DIR": "./ov_cache"        # Cache compiled model, avoids recompiling
    }
)

# OpenVINO will silently fall back to CPU for unsupported ops
# Check the compiled model's device assignment
compiled = model.model  # Access the underlying CompiledModel
print(compiled.get_property("EXECUTION_DEVICES"))
# Should show: NPU or NPU,CPU (heterogeneous — normal for some ops)

tokenizer = AutoTokenizer.from_pretrained(model_path)

inputs = tokenizer("Write a poem about Python programming:", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


`dynamic_shapes` was set to False, but this value will be ignored as only dynamic shapes are supported.


RuntimeError: Exception from src\inference\src\cpp\core.cpp:113:
Exception from src\inference\src\dev\plugin.cpp:53:
Exception from src\plugins\intel_npu\src\plugin\src\plugin.cpp:879:
Exception from src\plugins\intel_npu\src\compiler_adapter\src\ze_graph_ext_wrappers.cpp:405:
L0 pfnCreate2 result: ZE_RESULT_ERROR_INVALID_ARGUMENT, code 0x78000004 - generic error code for invalid arguments . [NPU_VCL] Compiler returned msg:
Got negative shape dim bound: '-1'



